# randn-like-noise-source — faded example 3: Audit reparameterization for dtype leak

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `randn-like-noise-source`. Running the beacon reports progress on the `Generative: randn-like noise source` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Generative: randn-like noise source` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`randn-like-noise-source`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "randn-like-noise-source"
DD_SUBTOPIC = "Generative: randn-like noise source"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

A dtype 'leak' in reparameterization occurs when the noise `eps` is sampled with `torch.randn(*sigma.shape)` instead of `torch.randn_like(sigma)`. The output `z` inherits `float32` from `eps` even when `sigma` is `float64`, silently losing precision. An audit function checks whether `z.dtype` matches the input `sigma.dtype` for each test case.

## Faded exercise 3

### Exercise — Audit reparameterization for dtype leak

Complete `audit_dtype(reparam_fn, dtypes)`. For each dtype, build `mu` and `sigma`, call `reparam_fn`, and record `(dtype, z.dtype, leaked)` where `leaked = (z.dtype != dtype)`.

Fill in the `leaked` computation.

**Fill in:** Compute the leaked boolean as True if z's dtype does not match the expected input dtype.

In [ ]:
import torch as t

def audit_dtype(reparam_fn, dtypes):
    report = []
    for dtype in dtypes:
        mu = t.zeros(4, 8, dtype=dtype)
        sigma = t.ones(4, 8, dtype=dtype)
        z = reparam_fn(mu, sigma)
        leaked = None  # TODO: Compute the leaked boolean as True if z's dtype does not match the expected input dtype.
        report.append((dtype, z.dtype, leaked))
    return report

def reparam_correct(mu, sigma):
    return mu + sigma * t.randn_like(sigma)

def reparam_buggy(mu, sigma):
    return mu + sigma * t.randn(*sigma.shape)  # bug: always float32

t.manual_seed(35)
dtypes = [t.float32, t.float64]
print('Correct:', audit_dtype(reparam_correct, dtypes))
print('Buggy:  ', audit_dtype(reparam_buggy, dtypes))


def _test():
    import torch as t

    def reparam_correct(mu, sigma):
        return mu + sigma * t.randn_like(sigma)

    def reparam_buggy(mu, sigma):
        return mu + sigma * t.randn(*sigma.shape)

    dtypes = [t.float32, t.float64]

    # Correct function: no leaks
    correct_report = audit_dtype(reparam_correct, dtypes)
    for dtype, z_dtype, leaked in correct_report:
        assert not leaked, f'correct reparam should not leak on {dtype}'

    # Buggy function: leaks on float64
    buggy_report = audit_dtype(reparam_buggy, dtypes)
    float64_entry = [r for r in buggy_report if r[0] == t.float64][0]
    assert float64_entry[2] is True, 'buggy reparam should leak on float64'


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

def audit_dtype(reparam_fn, dtypes):
    report = []
    for dtype in dtypes:
        mu = t.zeros(4, 8, dtype=dtype)
        sigma = t.ones(4, 8, dtype=dtype)
        z = reparam_fn(mu, sigma)
        leaked = (z.dtype != dtype)
        report.append((dtype, z.dtype, leaked))
    return report

def reparam_correct(mu, sigma):
    return mu + sigma * t.randn_like(sigma)

def reparam_buggy(mu, sigma):
    return mu + sigma * t.randn(*sigma.shape)  # bug: always float32

t.manual_seed(35)
dtypes = [t.float32, t.float64]
print('Correct:', audit_dtype(reparam_correct, dtypes))
print('Buggy:  ', audit_dtype(reparam_buggy, dtypes))
```
</details>